In [2]:
import pandas as pd
import numpy as np
from pickle import dump
from scipy.stats import randint, uniform

# Feature Selection
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from catboost import CatBoostRegressor
from sklearn.model_selection import ParameterSampler

## Preparación de datos por semanas

>Ordeno el dataset por `num_semana` y separo las semanas en bloques para mantener el orden temporal.  
>Luego dejo listo `X` (features) y `y` (objetivo) para entrenar el modelo.


In [3]:
seed = 18

#df = pd.read_csv("../data/processed/df_v2")
#df = pd.read_csv("../data/processed/df_withinetacolum")
df = pd.read_csv("../data/processed/df")


df = df.sort_values("num_semana").reset_index(drop=True)
weeks = df["num_semana"].unique()

cut_w = int(len(weeks) * 0.8)
train_weeks = weeks[:cut_w]
test_weeks  = weeks[cut_w:]

train = df[df["num_semana"].isin(train_weeks)]
test  = df[df["num_semana"].isin(test_weeks)]

X_train, y_train = train.drop(columns=["y"]), train["y"]
X_test,  y_test  = test.drop(columns=["y"]),  test["y"]


In [4]:
# Crear validation desde el train (por semanas)
weeks_train = train["num_semana"].unique()
cut_val = int(len(weeks_train) * 0.9)   # 90% train interno, 10% val (ajusta si quieres)

tr_weeks  = weeks_train[:cut_val]
val_weeks = weeks_train[cut_val:]

train_tr  = train[train["num_semana"].isin(tr_weeks)]
train_val = train[train["num_semana"].isin(val_weeks)]

X_tr, y_tr   = train_tr.drop(columns=["y"]),  train_tr["y"]
X_val, y_val = train_val.drop(columns=["y"]), train_val["y"]


In [5]:
cb = CatBoostRegressor(
    loss_function="RMSE",
    random_seed=18,
    iterations=5000,
    learning_rate=0.05,
    depth=8,
    verbose=0,
    allow_writing_files=False
)

cb.fit(
    X_train, y_train,
    cat_features=["product"],
    eval_set=(X_val, y_val),
    early_stopping_rounds=200,
    use_best_model=True
)

CatBoostRegressor(allow_writing_files=False, depth=8, iterations=5000, learning_rate=0.05, loss_function='RMSE', random_seed=18, verbose=0)

In [13]:
pred_test = cb.predict(X_test)
pred_train = cb.predict(X_train)
mse_test  = mean_squared_error(y_test, pred_test)
rmse_test = np.sqrt(mse_test)
r2_test   = r2_score(y_test, pred_test)
r2_train = r2_score(y_train, pred_train)

print(f"MSE (Test): {mse_test:.2f}")
print(f"RMSE (Test): {rmse_test:.2f}")
print(f"R² (Train): {r2_train:.2f}")
print(f"R² (Test): {r2_test:.2f}")

MSE (Test): 28.83
RMSE (Test): 5.37
R² (Train): 0.98
R² (Test): 0.63


In [7]:
# suponiendo que ya tienes train df (solo semanas train) como antes
weeks_train = train["num_semana"].unique()
cut_val = int(len(weeks_train) * 0.9)

tr_weeks  = weeks_train[:cut_val]
val_weeks = weeks_train[cut_val:]

train_tr  = train[train["num_semana"].isin(tr_weeks)]
train_val = train[train["num_semana"].isin(val_weeks)]

X_tr, y_tr   = train_tr.drop(columns=["y"]), train_tr["y"]
X_val, y_val = train_val.drop(columns=["y"]), train_val["y"]


In [8]:
rng = np.random.RandomState(seed)

param_dist = {
    "iterations": randint(800, 5000),
    "learning_rate": uniform(0.01, 0.09),
    "depth": randint(4, 11),
    "l2_leaf_reg": uniform(1, 9),
    "random_strength": uniform(1, 4),
    "bagging_temperature": uniform(0, 1),
    "border_count": [128],
}

best_score = -1e9
best_params = None
best_model = None

for params in ParameterSampler(param_dist, n_iter=50, random_state=rng):
    model = CatBoostRegressor(
        loss_function="RMSE",
        random_seed=seed,
        verbose=0,
        allow_writing_files=False,
        **params
    )

    model.fit(
        X_tr, y_tr,
        cat_features=["product"],
        eval_set=(X_val, y_val),
        early_stopping_rounds=200,
        use_best_model=True
    )

    preds = model.predict(X_val)
    score = r2_score(y_val, preds)

    if score > best_score:
        best_score = score
        best_params = params
        best_model = model

print("Mejor R2 (val):", best_score)
print("Mejores params:", best_params)


Mejor R2 (val): 0.7708935265067547
Mejores params: {'bagging_temperature': np.float64(0.6869701913398437), 'border_count': 128, 'depth': 8, 'iterations': 4397, 'l2_leaf_reg': np.float64(2.787709232654028), 'learning_rate': np.float64(0.06914701275493802), 'random_strength': np.float64(3.7986225159683804)}


In [11]:
pred_test = best_model.predict(X_test)
pred_train = best_model.predict(X_train)
mse_test  = mean_squared_error(y_test, pred_test)
rmse_test = np.sqrt(mse_test)
r2_test   = r2_score(y_test, pred_test)
r2_train = r2_score(y_train, pred_train)

print(f"MSE (Test): {mse_test:.2f}")
print(f"RMSE (Test): {rmse_test:.2f}")
print(f"R² (Train): {r2_train:.2f}")
print(f"R² (Test): {r2_test:.2f}")

MSE (Test): 22.15
RMSE (Test): 4.71
R² (Train): 0.77
R² (Test): 0.72


In [12]:
dump(best_model, open("../models/72_Cat_Boost_Regressor.pkl", "wb"))

## Conclusión del modelo (CatBoost)

Tras el proceso de ajuste de hiperparámetros, el modelo final alcanzó un **R² de 0.72 en el conjunto de test**, frente a un **R² de 0.77 en entrenamiento**.

El **RMSE en test fue de 4.71**, mostrando un rendimiento consistente en semanas posteriores no utilizadas para el entrenamiento. La diferencia entre el rendimiento de train y test es relativamente moderada, aunque indica cierto descenso de rendimiento al generalizar a datos futuros.

En conjunto, **CatBoost mostró un buen equilibrio entre capacidad predictiva y generalización** para este problema de forecasting y fue seleccionado como modelo final.

El modelo entrenado se guarda en formato `.pkl` para su reutilización en el pipeline de predicción y en la aplicación.